## Ridge Regression (Closed Form)

### 1. Problem Setup

Assume we have a dataset:

$$
X \in \mathbb{R}^{N \times D}, \quad y \in \mathbb{R}^{N}
$$

where  
- $X$ is the feature matrix  
- $y$ is the target vector  

The goal is to learn a function that predicts $y$ from $X$.

---

### 2. Linear Model

We assume a linear relationship:

$$
y = X \theta
$$

where  
- $\theta \in \mathbb{R}^{D}$ are the model parameters.

---

### 3. Adding the Bias Term

To include an intercept, we augment the feature matrix:

$$
X' =
\begin{bmatrix}
1 & x_1 \\
1 & x_2 \\
\vdots \\
1 & x_N
\end{bmatrix}
$$

Now,

$$
y = X' \theta
$$

---

### 4. Ordinary Least Squares (OLS)

The standard linear regression solution minimizes:

$$
\| y - X\theta \|^2
$$

Closed-form solution:

$$
\theta = (X^T X)^{-1} X^T y
$$

---

### 5. Ridge Regression Idea

To prevent overfitting, we add an $L_2$ penalty:

$$
\| y - X\theta \|^2 + \lambda \|\theta\|^2
$$

where  
- $\lambda$ controls regularization strength.

---

### 6. Objective Function

$$
J(\theta) = (y - X\theta)^T (y - X\theta) + \lambda \theta^T \theta
$$

---

### 7. Closed Form Solution

Taking derivative and setting to zero:

$$
\theta = (X^T X + \lambda I)^{-1} X^T y
$$

---

### 8. Bias Not Regularized

When bias is included:

$$
I =
\begin{bmatrix}
0 & 0 & \dots \\
0 & 1 & \dots \\
\vdots & & \ddots
\end{bmatrix}
$$

This ensures the **bias term is not penalized**.

---

### 9. Prediction Rule

For a new input $x$:

$$
\hat{y} = x^T \theta
$$

---

### 10. Effect of Regularization $\lambda$

- **Small $\lambda \approx 0$**:
  - Model behaves like standard linear regression
  - Fits training data closely
  - Risk of overfitting

- **Large $\lambda$**:
  - Coefficients shrink toward zero
  - Model becomes simpler
  - Predictions are smoother
  - Risk of underfitting

---

### 11. Geometric Interpretation

Ridge regression constrains the solution:

$$
\|\theta\|^2 \leq c
$$

Instead of an unconstrained solution, we search inside a **sphere**, which stabilizes the model.

---

### 12. Optimization Insight

Ridge regression improves numerical stability by making:

$$
X^T X + \lambda I
$$

always invertible, even when $X^T X$ is singular.

---

### 13. Algorithm Summary

- Add bias term (optional)

Compute:

$$
\theta = (X^T X + \lambda I)^{-1} X^T y
$$

Predict using:

$$
\hat{y} = X \theta
$$

---

### 14. Final Perspective

Ridge regression solves:

$$
\min_{\theta} \; \|y - X\theta\|^2 + \lambda \|\theta\|^2
$$

It balances:
- **Data fitting**  
- **Model simplicity**

---



In [1]:
class RidgeRegressionClosedForm:
    def __init__(self, lmbda=0.1, add_bias=True):
        self.lmbda = lmbda
        self.add_bias = add_bias
        self.theta = None

    def fit(self, X, y):
        self.theta = None

        X = np.asarray(X)
        y = np.asarray(y).reshape(-1)

        if X.ndim == 1:
            X = X.reshape(-1, 1)

        N, D = X.shape
        if self.add_bias:
            X = np.hstack((np.ones((N, 1)), X))
            D += 1

        identity = np.eye(D)

        if self.add_bias:
            identity[0, 0] = 0

        self.theta = np.linalg.inv(X.T @ X + self.lmbda * identity) @ X.T @ y
        print(f'The model weights are :  {self.theta}')

        return 

    def predict(self, X):
        if self.theta is None:
            raise ValueError("Model is not fitted yet.")

        X = np.asarray(X)

        if X.ndim == 1:
            X = X.reshape(-1, 1)

        N = X.shape[0]

        if self.add_bias:
            X = np.hstack((np.ones((N, 1)), X))

        return X @ self.theta

## Case 1: Perfect Linear Data

- The data follows an exact linear relationship.

- **Small $\lambda$**:
  - Model fits data almost perfectly.
  - Equivalent to ordinary least squares.

- **Large $\lambda$**:
  - Coefficients shrink toward zero.
  - Predictions are slightly underestimated.

$$
\text{Effect: Increasing } \lambda \Rightarrow \text{simpler model, higher bias}
$$

In [2]:
import numpy as np

X = np.array([1, 2, 3, 4])
y = np.array([2, 4, 6, 8])

In [3]:
model = RidgeRegressionClosedForm(lmbda=0.01)
model.fit(X, y)
preds = model.predict(X)
print(f'Predictions on training data: {preds}')

The model weights are :  [0.00998004 1.99600798]
Predictions on training data: [2.00598802 4.00199601 5.99800399 7.99401198]


In [4]:
model = RidgeRegressionClosedForm(lmbda=5)
model.fit(X, y)
preds = model.predict(X)
print(f'Predictions on training data: {preds}')

The model weights are :  [2.5 1. ]
Predictions on training data: [3.5 4.5 5.5 6.5]


---

## Case 2: Collinear Features

- The second column is a multiple of the first.

$$
X^T X \text{ is singular}
$$

- **OLS fails** due to non-invertibility.
- Ridge regression works because:

$$
X^T X + \lambda I
$$

is always invertible.

- **Effect of $\lambda$**:
  - Stabilizes the solution
  - Prevents numerical instability

In [5]:
X = np.array([
    [1, 2],
    [2, 4],
    [3, 6]
])
y = np.array([1, 2, 3])

In [6]:
model = RidgeRegressionClosedForm(lmbda=0.1)
model.fit(X, y)
print(f'Predictions on training data: {model.predict(X)}')

The model weights are :  [0.01980198 0.1980198  0.3960396 ]
Predictions on training data: [1.00990099 2.         2.99009901]


---

## Case 3: Noisy Data

- Data contains small random noise.

- **Small $\lambda$**:
  - Model fits noise closely
  - High variance

- **Large $\lambda$**:
  - Smooths predictions
  - Ignores small fluctuations

$$
\text{Effect: Larger } \lambda \Rightarrow \text{noise reduction}
$$

In [7]:
X = np.array([1, 2, 3, 4, 5])
y = np.array([2.1, 3.9, 6.2, 7.8, 10.5])

In [8]:
model_small = RidgeRegressionClosedForm(lmbda=0.01)
print("Model (λ = 0.01)")

model_small.fit(X, y)

model_large = RidgeRegressionClosedForm(lmbda=5)
print("\nModel (λ = 5)")

model_large.fit(X, y)

print("\nPredictions with Small Lambda (λ = 0.01)")
print(np.round(model_small.predict(X), 3))

print("\nPredictions with Large Lambda (λ = 5)")
print(np.round(model_large.predict(X), 3))

Model (λ = 0.01)
The model weights are :  [-0.1037962   2.06793207]

Model (λ = 5)
The model weights are :  [1.96 1.38]

Predictions with Small Lambda (λ = 0.01)
[ 1.964  4.032  6.1    8.168 10.236]

Predictions with Large Lambda (λ = 5)
[3.34 4.72 6.1  7.48 8.86]


---

## Case 4: High-Dimensional Data

- Number of features exceeds number of samples:

$$
D > N
$$

- Infinite solutions exist in OLS.
- Ridge provides a unique solution.

- **Effect of $\lambda$**:
  - Controls magnitude of coefficients
  - Ensures stability

$$
\text{Ridge ensures a unique and stable solution}
$$

In [9]:
X = np.array([
    [1, 2, 3, 4],
    [2, 3, 4, 5],
    [3, 4, 5, 6]
])
y = np.array([1, 2, 3])

In [10]:
model = RidgeRegressionClosedForm(lmbda=1,add_bias=False)
model.fit(X, y)
print(f'Predictions on training data: {model.predict(X)}')

The model weights are :  [0.33676976 0.23367698 0.13058419 0.02749141]
Predictions on training data: [1.30584192 2.03436426 2.7628866 ]


---

## Case 5: Zero Variance Feature

- Second feature has constant value.

- It carries no useful information.

- Ridge regression shrinks its coefficient:

$$
\theta_j \rightarrow 0
$$

- Model focuses on informative features.

$$
\text{Effect: Ridge performs implicit feature selection via shrinkage}
$$

In [11]:
X = np.array([
    [1, 5],
    [2, 5],
    [3, 5],
    [4, 5]
])
y = np.array([2, 4, 6, 8])

In [12]:
model = RidgeRegressionClosedForm(lmbda=0.01,add_bias=False)
model.fit(X, y)
print(f'Predictions on training data: {model.predict(X)}')

The model weights are :  [1.99600997e+00 1.99481309e-03]
Predictions on training data: [2.00598404 4.00199402 5.99800399 7.99401396]
